<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/03-dataframe-fundamentals/00-creating-dataframes-and-schemas.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 3.0 — Creating DataFrames and schemas

The setup cell defines `ORDERS_SCHEMA` and the `orders` DataFrame — every
Module 3 notebook starts from this same pattern.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("3.0-creating-dataframes")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_CSV = f"{DATA_DIR}/orders.csv"

## From a collection (tests, tiny lookups)

In [ ]:
tiny = spark.createDataFrame(
    [("c001", "Priya", 34), ("c002", "Jonas", 41)],
    schema="customer_id STRING, name STRING, age INT",
)
tiny.show()
tiny.printSchema()

## From an RDD (the Module 2 bridge)

In [ ]:
sc = spark.sparkContext
rdd = sc.parallelize([("c001", 3), ("c002", 1)])
rdd.toDF(["customer_id", "orders"]).show()

# and back again: DataFrame -> RDD of Row objects
tiny.rdd.take(2)

## From a file, with inference — watch the Jobs tab!

Check the Jobs tab count BEFORE and AFTER this cell: `inferSchema` runs a
real job even though we called no action. Then note what types it guessed
(your `order_date` may be `date` or `string` depending on Spark version —
that variability is part of the lesson).

In [ ]:
inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)
inferred.printSchema()

## The production way: explicit schema, no inference job

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])

orders = spark.read.option("header", True).schema(ORDERS_SCHEMA).csv(ORDERS_CSV)
orders.printSchema()   # stable types, zero extra jobs
orders.show(5)

In [ ]:
# Inspection tools
print(orders.columns)
print(orders.dtypes)
print("partitions:", orders.rdd.getNumPartitions())   # still partitioned underneath

## Exercises

1. Count the nulls you can spot in `orders.show(41)` — which columns have them? (Cross-check `data/README.md`.)
2. Build a DataFrame from `spark.createDataFrame` where one row has `None` for age. What does `printSchema` say about nullability?
3. Change `quantity` in `ORDERS_SCHEMA` to `DoubleType()` and re-read. What do the values look like? Now try `DateType()` — what happens, and WHEN does it happen (read time or action time)?

In [ ]:
# your exercise code here
